# Cassette fronts: R33 native training comparison

**Upload this notebook to Colab, choose an L4 or A100 GPU, select Run all, and return `baomai-front-review-r33-results.zip`.**

One question: **do the verified R33 training-boundary corrections improve localization under the unchanged native loss?** The reference uses the previous R31 training labels; the candidate uses **8 corrected rows across 7 training photographs**. Every other training setting is shared.

PR113 was the second-seed check of doubled partial-box loss. It did not meet the prespecified clean-improvement screen: on the frozen 666-object reference, misses changed from 32 to 33 and corner-pair successes within 5 pixels changed from 524 to 522, although unmatched outputs fell from 105 to 100. The weight escalation stops here. This notebook uses the **unmodified native Pose26 loss in both arms**.

Both arms start independently from original epoch 173. Each receives 137 images once per epoch, including 13 empty scenes: **30 epochs, 1,050 batches, 4,110 image presentations and 78 optimizer updates**, seed 46, batch 4, image size 1280, FP32, AdamW and initial learning rate 0.0001. Final-epoch selection is fixed in advance.

The five code cells fetch the published source tree, recreate the original runtime commit identity, restore original inputs, check the runtime, run a discarded four-batch resource trial and the matched pair, then package the return ZIP. Completed runs are verified before reuse and backed up to Drive when available.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REVISION = 'd4b905dc0ed7728815108fed04486bc4215f7a2b'
PROTOCOL_PATH = 'experiments/front-review-2026-11-25-r33.json'
PROTOCOL_SHA256 = '9782658c99d56740af0302c4edd80a071e169ca404c31b01948c7a2092d078ed'
EXPERIMENT_ID = 'front-review-r33'
ROLE_RECIPES = {'reference': 'reviewed-incremental-fronts-seed46-v1', 'candidate': 'reviewed-incremental-fronts-seed46-v1'}
RECIPE_SHA256 = 'f18e0a813a1057e1b3ab3a44ea3ce9b2a3e7673cc2bf04ca09b394bfdd8c51b8'
RUN_BASE = (Path("/content") if Path("/content").exists() else Path.cwd()) / "baomai-front-review-r33"
ROLES = ('reference', 'candidate')

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the final handoff notebook with its full source commit pinned."
assert re.fullmatch(r"[0-9a-f]{64}", PROTOCOL_SHA256), "Use the final handoff notebook with its protocol hash pinned."
assert sys.version_info >= (3, 11), "Python3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

workspace_identity = {"source_revision": SOURCE_REVISION, "protocol_sha256": PROTOCOL_SHA256}
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    assert RUN_BASE.is_dir() and not RUN_BASE.is_symlink(), "Choose a fresh nonsymlink workspace."
    marker = RUN_BASE / "workspace-identity.json"
    assert marker.is_file() and json.loads(marker.read_text()) == workspace_identity, "Existing workspace belongs to different pins; preserve it and choose a new RUN_BASE."
else:
    RUN_BASE.mkdir(parents=True)
    (RUN_BASE / "workspace-identity.json").write_text(json.dumps(workspace_identity, indent=2) + "\n")
REPO = RUN_BASE / "dl"
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Drive is unavailable here; preserve completed bundles and the ZIP from the local workspace.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-front-review-r33-20261125")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def safe_member(name):
    p = Path(name)
    assert isinstance(name, str) and name and not p.is_absolute() and ".." not in p.parts
    assert "\\" not in name and ":" not in name and p.as_posix() == name
    return p

def artifact_inventory(path):
    path = Path(path)
    if path.is_file():
        names = [path.name]
        base = path.parent
    else:
        base = path
        receipt = json.loads((path / "training-receipt.json").read_text())
        names = ["training-receipt.json", "staging-manifest.json", receipt["log"]["path"]]
        if (path / "checkpoint-manifest.json").is_file():
            manifest = json.loads((path / "checkpoint-manifest.json").read_text())
            names += ["checkpoint-manifest.json", manifest["checkpoint"]["path"], receipt["results"]["path"]]
    assert len(names) == len(set(names))
    return [{"path": name, "sha256": sha256(base / safe_member(name)), "size_bytes": (base / name).stat().st_size}
            for name in sorted(names)]

def validate_copy(path, marker):
    base = path if marker["directory"] else path.parent
    for entry in marker["files"]:
        target = base / safe_member(entry["path"])
        assert target.is_file() and not target.is_symlink(), f"Missing backup artifact: {target}"
        assert target.stat().st_size == entry["size_bytes"] and sha256(target) == entry["sha256"], f"Backup hash mismatch: {target}"

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    relative = path.relative_to(RUN_BASE).as_posix()
    if relative == "resource-trial":
        relative = "resource-trials/" + sha256(path / "training-receipt.json")[:20]
    destination = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(destination) + ".complete")
    marker = {**workspace_identity, "relative_path": relative, "directory": path.is_dir(), "files": artifact_inventory(path)}
    if marker_path.is_file():
        assert json.loads(marker_path.read_text()) == marker, f"Drive already has different completed evidence: {destination}"
        validate_copy(destination, marker)
        print("Verified existing Drive backup:", destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Only an incomplete copy in this notebook's exact destination is replaced.
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    if path.is_dir():
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path / entry["path"], target)
    else:
        shutil.copyfile(path, destination)
    validate_copy(destination, marker)
    marker_path.write_text(json.dumps(marker, indent=2) + "\n")
    print("Saved verified completed artifacts:", destination, flush=True)

def restore_from_drive(relative):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(source) + ".complete")
    if not marker_path.is_file():
        return None
    marker = json.loads(marker_path.read_text())
    assert all(marker.get(k) == v for k, v in workspace_identity.items()), "Drive completion belongs to different source/protocol pins."
    assert marker["relative_path"] == relative
    validate_copy(source, marker)
    destination = RUN_BASE / safe_member(relative)
    if destination.exists() or destination.is_symlink():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if marker["directory"]:
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / entry["path"], target)
    else:
        assert len(marker["files"]) == 1 and marker["files"][0]["path"] == source.name
        shutil.copyfile(source, destination)
    validate_copy(destination, marker)
    print("Restored completed evidence:", destination, flush=True)
    return destination

print("Workspace:", RUN_BASE)

BASE_REVISION = '337a12d8b11f9925b493cba794f29a4ffcf848ed'
SOURCE_URL = 'https://github.com/Siuuuers/dl.git'
PUBLISHED_SOURCE_REVISION = '1acb4061a99a4484be6226db5dafddd0cb17c103'
SOURCE_TREE = 'd3ae26eebf73872a34ff722b9d6c6bc6c8527e69'
BASE_HELPER_SHA256 = {'scripts/build_front_review_r10.py': '37ff963e6068604698595ce363b50df4215157519ec881558afc8ff94b76ecef', 'scripts/build_front_review_r11.py': 'be37437dea7e2298a4247d328f472dab1a808854a392f987e78fa97578d35c45', 'scripts/build_front_review_r12.py': 'f372b76d3bbfd614201518720cefd428993e9de58a5958ea227d9001951453d3', 'scripts/build_front_review_r13.py': 'e2ced3f074fb56fc2fbf82bf42cb8e32cdbc75b8569231775aa454758ecc7d02', 'scripts/build_front_review_r14.py': 'd561f7694b44367b25c5ec4dce460b5c22f3a58c017ee6de3d99e5bb11ed00de', 'scripts/build_front_review_r15.py': '9cbb731008484b4d610acd59b6793c74440270ce007e03d60d72f42bd8316f03', 'scripts/build_front_review_r16.py': 'fe2ee797e4d1506062689de839e700bf8bd307047f82221639539112aa92ba28', 'scripts/build_front_review_r17.py': '52c7a842b70a3697ee3309a14b012c8ab8028763b26b620da5588b3225eabe23', 'scripts/build_front_review_r18.py': '7ef784c3359d4d61ff1aecda15bb310b2ea3c431ed67e0959afc7183d1c1f0c9', 'scripts/build_front_review_r19.py': '473648dfdc14cebf34f6112aaa746cdab0469749fef2b3911ea530f092b8185d', 'scripts/compose_incremental_pose_candidate.py': '9b51b8a96407af15fa820b959d30ccffe3da21ca53b4ba1f91294bb46360a789', 'scripts/compose_observed_pose_review.py': '8f99c87236b09ed40c85521b68ea2a39ca4062cfa74b8cd43bb376cb5009fd25', 'scripts/evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'scripts/front_review_experiment.py': '58fc3f020edfe834c4d051ba98a954661ffaf7bdf2ce2dbc0caf2fbec3a7f524', 'scripts/front_review_r20.py': 'f7817a4f90876e4e17b6c0902fb90b067447cd1fa5d5195c15e5f737a63e4746', 'scripts/front_review_r21.py': 'f27176d48a7f34be4e533665792cac2a1185030349f4ba7a98774b060dd606fe', 'scripts/front_review_r22.py': 'cbb617b7be4e418d85c354780356119cda823557e5deafaddf5c31b2e30c5d76', 'scripts/front_review_r23.py': '626106c6e63f90af1a427f0b14b7c4f391ec94965fae5806b4d2dcb47aebe687', 'scripts/front_review_r24.py': '1f752d0d31799e2b824ed9eadea9618d258d27be0d4ddb55186fe3c84b526f9d', 'scripts/front_review_r25.py': 'fa2823f38f5344df3e26f97b98fe6f0565bb64ebf3d6fcec8a518f2ebf1b0a18', 'scripts/front_review_r26.py': 'cf0433f7537512a4f52317c3ac6180a78bdb7c37f4353ddad3ee221e19703d07', 'scripts/front_review_r27.py': '9525313fd817ac7177daf73e07da951924ab862112bbf0a5870225aa649d7a07', 'scripts/front_review_r28.py': '8cee625eff3c8c49064901e26fbf56f90ba8e65808ecf99d935ef95bf874c0a5', 'scripts/front_review_r28_training.py': '8426f8fbcd3e872eb5ed58e88dc86cfb9788223830b3ffb0a42c9d768a016fa0', 'scripts/front_review_r30.py': '27d0967af3c3291d26dc388e5eca0798474fe5cbcc42b1fe730a5758c11a4ca3', 'scripts/package_gpu_results.py': '120b1a3d22f9cb8896af67a84b7d3da686d89e41b63d5889eb7fb106bdce477c', 'scripts/partial_target_loss.py': '1cc1f16acf1bc2a19b12927da553ab67a3fe819f14e093620f90a1128b29569f', 'scripts/pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'scripts/predict_pose_folder.py': '239c3b1dd7f5fefc46b8b102277d86aec53d7e92dff99b890eaccfd74adfdefb', 'scripts/refine_reconciled_pose_labels.py': 'a9cee6c7a341801a18f390ad281530a0936f210f177d15b2c6b5942b6c363e05', 'scripts/restore_gpu_inputs.py': 'dbe0e2658d9752e5dcd244f377f5f4ed4e18b63d5f52fc73ff3f1ddd85768aa0', 'scripts/train_pose_partial_loss.py': '31157f7359c0f4c2e85c58873597af6af564654d943294ac42e36da81e93596e', 'scripts/verify_front_review_r10.py': 'cc1fec49a90fcce865a55860986247a179d04dd61246e63bc043b343600f2d0d', 'scripts/verify_front_review_r11.py': '3836f9cc414d3d8d964e777e2b3cddf6b58fdfe23c618e67f1afbe1e4b93b023', 'scripts/verify_front_review_r12.py': '937fc14a80a1a805f483c73ce2a506c794ca17f501b56bd37285c69e2f1890a5', 'scripts/verify_front_review_r13.py': 'b9c00e1facf3017f59090aff6f50d0cced1ada1aa953ed9c666c2fdc9f44e668', 'scripts/verify_front_review_r14.py': 'b28d421ddcb7d5ffd54b3511335fe206dd422b4f67bc701a0e05c48b360c7b56', 'scripts/verify_front_review_r15.py': '79a2c550a68d1d9c6a9308fa362ac34f8e2a230a59913dd12a2e6d9fd810d48d', 'scripts/verify_front_review_r16.py': 'e6fe5938ddde0a91d603c12e3cc846463d10cdd1e2844867663d24046c0ce334', 'scripts/verify_front_review_r17.py': '8ae0320b5ad4bb099fe947bab60b1b38a7999721138a32bf3540f9f8f50d8b7a', 'scripts/verify_front_review_r18.py': 'f2ced60a852155896cc7830405a206eb3e459a2fd91d186e60906328b34f05b2', 'scripts/verify_front_review_r19.py': '83ce8e77b7d2b1e9ceed6b7bc7e5c56b27502fa08f15d766956360ee2800b2d6', 'scripts/verify_front_review_r4.py': 'b46393cb5a186fd687ed6e2a4e1141711e319e138f361bd6fcbe254effb0afa0', 'scripts/verify_front_review_r5.py': '74b0108ed6bcd109dae0cb278b23fd6cc8f44cbbfb20c65d4ce1fa9e53570d94', 'scripts/verify_front_review_r6.py': '2591cc62d1c9238971358d8d1a9f20f324684a4f125aa63b29a1326c4bd00c82', 'scripts/verify_front_review_r7.py': 'afc116835e371dd459bcf62274ed90124196cc7ccb15f580524322b0d7c85fee', 'scripts/verify_front_review_r8.py': '49675f497acb6fab08d473d506f30dea7b1fa20633bfc10a77c0c6ba03e64c30', 'scripts/verify_observed_training_candidate.py': '28fea1d483058988ea567d9382ed82f73e511cd9b692af7bb8c23c3463134426', 'scripts/verify_pose_annotation_candidate.py': '0dc3d263b4aad594e1019de8f8318dcd963d9c0c2f4604ad2dee408c83af7979', 'scripts/verify_pose_box_candidate.py': 'f8613b9b310f25930a9d47169650473d3fbc0a1032868dd80222d956d26e1ea5', 'scripts/verify_pose_candidate.py': '03395b9a82a4c120647a1cac51c89fd214d8cf9b15c04e2196a40f39077d8491', 'scripts/verify_pose_composed_candidate.py': 'e538dfce5de0b1fb10047bbce0598882ee1083825ed6061469c85a3daf08bd32', 'scripts/verify_pose_endpoint_candidate.py': '9adcf9805ba9ee6fbe252c05903bab5a997e9e7c5063362d77e1f5fcccdb53d5', 'scripts/verify_reconciled_pose_labels.py': '110de745439ce07f800f5a1dedbd6a4ec21d0f3d9dd6b878db107974d88f3d94'}
ASSEMBLY_NAME = 'R33 notebook assembly'
ASSEMBLY_EMAIL = 'r33-notebook@localhost'
ASSEMBLY_DATE = '2026-11-25T07:00:00+08:00'
ASSEMBLY_MESSAGE = 'Assemble verified R33 native supervision comparison locally'


## 1. Restore verified source and original inputs

Fetch the exact published R33 source tree from GitHub. Verify its tree hash and recreate the original runtime commit identity for compatibility with the completed PR114 run. No label files or source payloads are embedded. All 157 original photographs and the original initializer are verified. Interrupted or incompatible folders are preserved for inspection.


In [ ]:
def assemble_source(repo, origin=SOURCE_URL):
    repo = Path(repo)
    assert not repo.is_symlink(), f"Source folder is a symlink: {repo}. Choose a fresh RUN_BASE."
    assert not repo.exists() or repo.is_dir(), f"Preserve the existing file at {repo} and choose a fresh RUN_BASE."
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
    if not repo.exists():
        run(["git", "init", repo], env=git_env)
        run(["git", "remote", "add", "origin", origin], cwd=repo)
        actual_revision = None
    else:
        probe = subprocess.run(["git", "rev-parse", "--verify", "HEAD"], cwd=repo,
                               capture_output=True, text=True)
        assert probe.returncode == 0, (f"Source checkout is incomplete at {repo}. Preserve or rename only "
            "that source folder, then rerun. Keep the run and Drive folders intact.")
        actual_revision = probe.stdout.strip()
        assert actual_revision in (BASE_REVISION, PUBLISHED_SOURCE_REVISION, SOURCE_REVISION), (
            "Existing source belongs to different pins; preserve it and choose a new RUN_BASE.")
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), (
            "Existing source has local changes. Preserve or rename only the source folder; keep completed runs intact.")
    if actual_revision != SOURCE_REVISION:
        for revision in (PUBLISHED_SOURCE_REVISION, BASE_REVISION):
            run(["git", "fetch", "--depth=1", "origin", revision], cwd=repo, env=git_env)
        tree = subprocess.check_output(["git", "rev-parse", PUBLISHED_SOURCE_REVISION + "^{tree}"], cwd=repo, text=True).strip()
        assert tree == SOURCE_TREE, "Published source tree differs from the original R33 assembly."
        # Recreate only the original commit metadata, not its file contents.
        # This preserves PR114 receipts and completed-run recovery identity.
        source_commit_env = {**os.environ,
            "GIT_AUTHOR_NAME": ASSEMBLY_NAME, "GIT_AUTHOR_EMAIL": ASSEMBLY_EMAIL,
            "GIT_COMMITTER_NAME": ASSEMBLY_NAME, "GIT_COMMITTER_EMAIL": ASSEMBLY_EMAIL,
            "GIT_AUTHOR_DATE": ASSEMBLY_DATE, "GIT_COMMITTER_DATE": ASSEMBLY_DATE}
        reconstructed = subprocess.check_output(["git", "-c", "commit.gpgsign=false", "commit-tree",
            tree, "-p", BASE_REVISION, "-m", ASSEMBLY_MESSAGE], cwd=repo, env=source_commit_env, text=True).strip()
        assert reconstructed == SOURCE_REVISION, "Original R33 commit identity could not be reconstructed."
        run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=repo, env=git_env)
    assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() == SOURCE_REVISION
    assert subprocess.check_output(["git", "rev-parse", "HEAD^{tree}"], cwd=repo, text=True).strip() == SOURCE_TREE
    assert subprocess.check_output(["git", "rev-parse", "HEAD^"], cwd=repo, text=True).strip() == BASE_REVISION
    for relative, expected in BASE_HELPER_SHA256.items():
        assert sha256(repo / relative) == expected, f"Published helper differs: {relative}"
    assert sha256(repo / PROTOCOL_PATH) == PROTOCOL_SHA256
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip()
    print("Verified published R33 tree and original runtime source:", SOURCE_REVISION)
    return SOURCE_REVISION


def verify_original_inputs(repo):
    repo = Path(repo)
    manifest = json.loads((repo / "data/manifest.json").read_text())
    protocol = json.loads((repo / PROTOCOL_PATH).read_text())
    assert len(manifest["records"]) == 157
    for record in manifest["records"]:
        path = repo / "data" / safe_member(record["image"])
        assert sha256(path) == record["image_sha256"], f"Original image differs: {record['id']}"
    assert manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
    assert sha256(repo / manifest["model_checkpoint"]) == protocol["initial_checkpoint_sha256"]
    assert subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo).returncode == 0
    # Refresh only verified original LFS paths. Their pointer objects and the
    # committed source must stay identical; no changed inputs are committed.
    run(["git", "add", "--renormalize", "--", "data/images", manifest["model_checkpoint"]], cwd=repo)
    assert subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo).returncode == 0, (
        "Restored originals changed indexed pointer objects. Preserve this source folder for inspection.")
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), (
        "Source must remain clean after verified original-input restoration.")
    print("Verified 157 original images and the unchanged epoch173 initializer.")
    return {"images_verified": 157, "initializer_sha256": protocol["initial_checkpoint_sha256"]}

lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
assemble_source(REPO)

# Original images and fixed initializer (published data only).
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"], cwd=REPO, text=True).splitlines())
assert len(expected_images) == 157 and tracked_images == expected_images
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt", "--exclude="], cwd=REPO)
verify_original_inputs(REPO)

## 2. Verify the exact treatment

The comparison changes only the accepted boundaries in 8 eligible training rows. All 137 training photographs remain, with 3,784 objects and 13 empty scenes. Both arms retain prior supervision for held photograph 015 and the 31 old-series photographs. Original trainer validation remains 20 images with 664 objects.

The complete R33 annotation review is separate from training eligibility. Its development-label amendments never enter training or trainer validation. The current R33 development reference is exploratory; frozen 666 and historical 663 remain unchanged.


In [ ]:
import torch
print("Runtime:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (completed-pair recovery only)")
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")), encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__)"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Limits: 600 seconds for the four-batch trial; 21600 seconds for each complete run.")


sys.dont_write_bytecode = True
sys.path.insert(0, str(REPO / "scripts"))
from evaluate_pose import preflight_dataset
from stage_pose_training import source_state
from partial_box_loss_r31 import verify_native_sources
from native_front_review_r33 import load_protocol, source_identity, verify_run, verify_trial, package_pair
verify_native_sources()  # Verify pinned native code; no custom criterion is attached.
CONTEXT = load_protocol(REPO, PROTOCOL_PATH)
assert CONTEXT["sha256"] == PROTOCOL_SHA256
document = CONTEXT["document"]
assert document["experiment_id"] == EXPERIMENT_ID
assert document["recipe_sha256"] == RECIPE_SHA256
assert all(document["recipe_id"] == ROLE_RECIPES[role] for role in ROLES)
assert document["execution"]["loss"] == "unmodified_native_Pose26"
PREFLIGHT = preflight_dataset(REPO, require_weights=True)
assert PREFLIGHT["model_sha256"] == document["initial_checkpoint_sha256"]
STATE = source_state(REPO, REPO / document["candidate"]["path"])
for role in ROLES:
    assert STATE["counts"][role] == document["roles"][role]["counts"]
    assert STATE["counts"][role]["train"]["images"] == 137
    assert STATE["counts"][role]["train"]["empty_images"] == 13
    assert STATE["counts"][role]["val"]["objects"] == 664
DELTA = CONTEXT["candidate_verification"]["training_input_delta"]["report"]
assert DELTA["summary"]["changed_rows"] > 0
assert DELTA["summary"]["classes_visibility_preserved"] is True
DELTA_PATH = RUN_BASE / "training-input-delta.json"
delta_bytes = (REPO / document["training_input_delta"]["path"]).read_bytes()
assert not DELTA_PATH.exists() or DELTA_PATH.read_bytes() == delta_bytes
DELTA_PATH.write_bytes(delta_bytes)
save_to_drive(DELTA_PATH)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified training change:", DELTA["summary"])
print("Reference uses previous R31 labels; candidate uses the exact eligible R33 corrections.")
print("Native Pose26 loss in both arms. Missing landmarks remain unavailable supervision.")
print("Each arm: seed 46; 137 images once per epoch; 30 epochs; 4110 image presentations; 1050 batches; 78 optimizer updates.")
print("Batch 4, image size 1280, FP32, AdamW, initial learning rate 0.0001. Resource-trial weights are discarded.")


## 3. Run the matched native-loss pair

Use one GPU runtime for both arms. The four-batch trial verifies image loading, gradients, parameter updates and memory before full training; its weights are discarded. Each full arm starts fresh from original epoch 173. Changing loss, seed, image size, thresholds or checkpoint-selection rules would create a different experiment.


In [ ]:
def assert_finished(output, role, *, trial=False):
    bundle = (verify_trial(REPO, output, role, CONTEXT) if trial
              else verify_run(REPO, output, role, CONTEXT))
    receipt = bundle["receipt"] if "receipt" in bundle else bundle
    assert receipt["source"] == source_identity(REPO)
    assert receipt["runtime"]["device"].startswith("cuda:")
    return receipt

def assert_current_runtime(receipt, current, label):
    expected = receipt["runtime"]
    assert all(expected[k] == v for k, v in current.items() if k != "cudnn_version"), f"Current runtime differs from {label}; select a matching runtime before full training."
    assert expected["backend"]["cudnn_version"] == current["cudnn_version"], f"Current cuDNN differs from {label}."

def train_one(role, *, trial=False):
    folder = "resource-trial" if trial else role
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        try:
            receipt = assert_finished(output, role, trial=trial)
        except Exception as error:
            raise FileExistsError(f"Preserve incomplete or incompatible run {output}. Archive only this incomplete folder, then retry with the same workspace. {error}") from error
        print("Reusing verified completed run:", folder, flush=True)
        save_to_drive(output)
        return output, receipt
    command = [sys.executable, "-B", "scripts/train_pose_comparison.py", "--root", str(REPO),
               "--candidate", document["candidate"]["path"], "--recipe", ROLE_RECIPES[role], "--trial" if trial else "--train", "--arm", role,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting", folder, "with its verified label arm and native loss. Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}
    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass
    started, last_notice = time.monotonic(), -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; {progress.get('status')}; epochs {len(progress.get('completed_epochs', []))}/30; batches {progress.get('batches_completed', 0)}; updates {progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: verifying and staging inputs ({elapsed:.0f}s)", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed; preserve its log and receipt. No partial run is accepted.")
    except BaseException:
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = assert_finished(output, role, trial=trial)
    save_to_drive(output)
    return output, receipt

COMPLETED = {}
for role in ROLES:
    restore_from_drive(role)
    output = RUN_BASE / role
    if output.exists() or output.is_symlink():
        _, COMPLETED[role] = train_one(role)

TRIAL_RUNTIME = None
if len(COMPLETED) == 2:
    assert COMPLETED["reference"]["runtime"] == COMPLETED["candidate"]["runtime"]
    print("Both roles are complete; no training will repeat.")
else:
    assert HOST_RSS_LIMIT_MIB >= 6000, "Choose a runtime with more free host RAM."
    probe = r"""
import importlib.metadata as m, json, platform, torch
assert torch.cuda.is_available(), "Select a CUDA GPU runtime before starting an incomplete pair."
assert (torch.ones(1, device='cuda:0') * 2).item() == 2
print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,
'ultralytics':m.version('ultralytics'),'torch_cuda':torch.version.cuda,'platform':platform.platform(),
'device':'cuda:0','device_name':torch.cuda.get_device_name(0),'device_capability':list(torch.cuda.get_device_capability(0)),
'packages':{n:m.version(n) for n in ('torchvision','numpy','Pillow','opencv-python','psutil')},
'cudnn_version':torch.backends.cudnn.version()}))
"""
    current = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True, cwd=REPO))
    for role, receipt in COMPLETED.items():
        assert_current_runtime(receipt, current, f"completed {role}")
    TRIAL, trial_receipt = train_one("candidate", trial=True)
    assert_current_runtime(trial_receipt, current, "resource trial")
    TRIAL_RUNTIME = trial_receipt["runtime"]
    for role, receipt in COMPLETED.items():
        assert receipt["runtime"] == TRIAL_RUNTIME, f"Trial differs from completed {role}; no full role started."

REFERENCE_RUN, reference_receipt = train_one("reference")
if TRIAL_RUNTIME is not None:
    assert reference_receipt["runtime"] == TRIAL_RUNTIME, "Control runtime differs from trial."
CANDIDATE_RUN, candidate_receipt = train_one("candidate")
assert reference_receipt["runtime"] == candidate_receipt["runtime"], "The two roles use different runtimes."
assert reference_receipt["training_labels"]["label_manifest"] == document["control"]["manifest"]
assert candidate_receipt["training_labels"]["label_manifest"] == document["candidate"]["path"]
assert reference_receipt["training_labels"]["training_protocol"] == candidate_receipt["training_labels"]["training_protocol"]
assert [s["image_ids"] for s in reference_receipt["batch_samples"]] == [s["image_ids"] for s in candidate_receipt["batch_samples"]]
assert reference_receipt["optimizer_steps"] == candidate_receipt["optimizer_steps"] == 78
assert reference_receipt["batches_completed"] == candidate_receipt["batches_completed"] == 1050
assert reference_receipt["images_processed"] == candidate_receipt["images_processed"] == 4110
print("Both roles completed with the same training budget. Package the verified artifacts next.")

## 4. Return the verified result ZIP

Return `baomai-front-review-r33-results.zip` with both final checkpoints, raw logs, receipts, protocol and exact training delta. Packaging checks the actual bytes and equality of the two runtimes. A completed, compatible ZIP can be verified and downloaded again without retraining.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-front-review-r33-results.zip"
restore_from_drive(RESULTS_ZIP.relative_to(RUN_BASE).as_posix())
result = package_pair(REPO, REFERENCE_RUN, CANDIDATE_RUN, RESULTS_ZIP,
                      verify_existing=RESULTS_ZIP.exists(), protocol=PROTOCOL_PATH)
print(json.dumps(result, indent=2))
save_to_drive(RESULTS_ZIP)
try:
    from google.colab import files
except ImportError:
    print("Completed results:", RESULTS_ZIP)
else:
    files.download(str(RESULTS_ZIP))

## How the result will be judged

Compare both fresh arms and retained PR35 on frozen 666, historical 663, preserved R32 and amended R33, reporting each reference separately. Include misses, unmatched outputs, duplicates, groups with zero, one or two visible corners, empty scenes and both-corner successes at 5 and 10 native pixels with eligible misses kept in denominators. Inspect exact gained/lost rows and continuous distances; do not select a seed or epoch after seeing results.

Keep the current one-to-many inference path, confidence 0.13 and NMS 0.70. The exploratory NMS 0.50 check removed many duplicates but lost real partial fronts; the already-trained one-to-one head also worsened recall. These diagnostics do not justify a default switch. Hidden-corner estimates remain permitted, but they are unscorable against unavailable annotation slots and are never substituted into ground truth.

Keep PR35 as the retained selected checkpoint until a separate review supports changing it. Repeated development-image review is not independent evidence of reliability on new capture sessions. This notebook performs no automatic model promotion.

**Annotation standard:** box the observed frontal face and its own visible molded lip/rim. Exclude standalone strips belonging only to rack, jaw, hand or another cassette; preserve an AABB when owned fragments or an owned lip require its extrema. Use color as contextual evidence. Stored image y increases downward: top = stored ymin, bottom = stored ymax.
